# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


In [2]:
#constants
N_TEST_POSES = 100
TIP_ERROR_SPEC_MM = 2

In [3]:
from src.simulation import constants
print({k: getattr(constants, k) for k in dir(constants) if k.isupper()})

{'BODY_PLACEMENT_ANGLE_STD': 0.03577817531305904, 'BODY_PLACEMENT_POS_STD': 8.94454382826476, 'CAL_SENSOR_NOISE_STD': 0.002, 'CAL_SENSOR_RANGE_MM': 5.0, 'HAND_GUIDE_ANGLE_STD': 0.05, 'HAND_GUIDE_POS_STD': 1.7889087656529516, 'MARKER_MANUFACTURING_STD': 0.7155635062611807, 'SAMPLE_DELTA_T_SEC': 0.03, 'SENSOR_NOISE_STD': 0.02, 'TEST_POSE_ANGLE_STD': 0.02, 'TEST_POSE_POS_STD': 0.7155635062611807, 'TRACKER_JIGGLE_ANGLE_STD': 0.01, 'TRACKER_JIGGLE_POS_STD': 5.0, '_CHI3_95': 2.795}


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [4]:
ptr_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points, e.g. vct3(x, y, z)
    vct3(0, 0, 0),
    vct3(250, 0, 0),
    vct3(0, 190, 0),
    vct3(60, 95, 160)
]
ptr_tip_approx = vct3(0, 0, -50) #50mm below the marker cluster  # TODO: vct3, the tip's approximate position in the pointer's local frame


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [5]:
cal_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points
    vct3(0, 0, 0),
    vct3(300, 0, 0),
    vct3(0, 270, 0),
    vct3(90, 120, 180)
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [6]:
F_tracker_approx = Frame(Rot(), vct3(300, 300, -1500))  # TODO: Frame, tracker pose relative to the workspace
F_cal_approx = Frame(Rot(), vct3(300, 300, 300))  # TODO: Frame, calibration object pose relative to the workspace


## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [7]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


# #debug
# for name, placed in [("ptr_mb", ptr_placed), ("cal_mb", cal_placed)]:
#     nominal = np.array([np.array(a.vec).reshape(3) for a in placed.body.nominal_marker_positions])
#     actual = np.array([[a.x, a.y, a.z] for a in placed.body.get_actual_marker_positions()])
#     errs = np.linalg.norm(nominal - actual, axis=1)
#     print(f"{name} manufacturing error per marker (mm): {np.round(errs, 3)}")


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [8]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    TODO: implement.

    input:
        trk - Tracker to read with
        placed - PlacedBody whose body's nominal marker positions should be
            refined in place (via body.set_marker_positions(...))
        n_readings - int, readings to average per pass
        passes - int, how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """
    body = placed.body
    for i in range(passes):
        readings = []
        for j in range(n_readings):
            obs = simulation.sample_marker_bodies(trk, [placed])[body.name]         #one pose for the body, and position of the 4 markers in the tracker frame
            F_inv = obs.frame.inv()             #converts points from the tracker frame to the body frame
            #print(F_inv)

            this_reading = []
            for m in obs.markers:           #obs.markers = list of the markers
                b = m.p             #b = the marker position in the tracker frame
                a = F_inv * b       #a = the same marker positions in the body frame
                this_reading.append(np.array(a.vec).reshape(3))
            readings.append(this_reading)
        
        averages = np.mean(readings, axis = 0)
        #print(averages)
        body.set_marker_positions([vct3(*row) for row in averages])
        print(f"{body.name} pass {i + 1} done")



    #raise NotImplementedError("TODO: implement marker-body refinement (see markdown above)")
    



refine_marker_body(trk, ptr_placed, n_readings=200, passes=10)
refine_marker_body(trk, cal_placed, n_readings=200, passes=10)

# print("ptr_mb refined:\n", np.array([np.array(a.vec).reshape(3) for a in ptr_mb.nominal_marker_positions]))
# print("cal_mb refined:\n", np.array([np.array(a.vec).reshape(3) for a in cal_mb.nominal_marker_positions]))

ptr_mb pass 1 done
ptr_mb pass 2 done
ptr_mb pass 3 done
ptr_mb pass 4 done
ptr_mb pass 5 done
ptr_mb pass 6 done
ptr_mb pass 7 done
ptr_mb pass 8 done
ptr_mb pass 9 done
ptr_mb pass 10 done
cal_mb pass 1 done
cal_mb pass 2 done
cal_mb pass 3 done
cal_mb pass 4 done
cal_mb pass 5 done
cal_mb pass 6 done
cal_mb pass 7 done
cal_mb pass 8 done
cal_mb pass 9 done
cal_mb pass 10 done


## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [9]:
N_OBSERVATIONS = 50  # TODO

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    R_desired = Rot.rand_xyz(np.pi/3, np.pi/3, np.pi)  # TODO: vary the desired pointer orientation across observations
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 50 observations


In [10]:
valid_count = sum(1 for s in sensed_tip_list if s is not None)
print(f"Valid observations (M): {valid_count} / {N_OBSERVATIONS}")

Valid observations (M): 49 / 50


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [11]:
from matplotlib.pylab import cond


def compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list):
    """
    TODO: implement.

    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
            the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
            the tracker for each observation j (Fcal,j), same order/length
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
        cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
            (e.g. from the least-squares fit's residuals)
    """
    A_blocks = []
    B_blocks = []

    for i in range(len(F_ptr_list)):
        if sensed_tip_list[i] is None:
            continue  # skip this observation if the tip was not sensed

        F_ptr = F_ptr_list[i]
        F_cal = F_cal_list[i]
        p_sensed = sensed_tip_list[i]
        F_j = F_cal.inv() * F_ptr # pointer pose in the cal frame
        R_j = F_j.R.matrix          # 3x3 rotation
        p_j = F_j.p.vec.ravel()     # 3x1 column
        s_j = p_sensed.vec.ravel()

        A_blocks.append(R_j)
        B_blocks.append(s_j - p_j)

    A = np.vstack(A_blocks)  # 3N x 3
    B = np.concatenate(B_blocks)  # 3N x 1

    # #debug
    # cond = np.linalg.cond(A)
    # print(f"Condition number of A: {cond:.2f}")
    # #end debug

    p_tip = np.linalg.inv(A.T @ A) @ A.T @ B 

    residual = A @ p_tip - B
    num_unknowns = 3
    residual_variance = (residual @ residual) / (len(B) - num_unknowns)
    cov_estimate = residual_variance * np.linalg.inv(A.T @ A)

    return vct3(*p_tip), cov_estimate

                                       
        
    # raise NotImplementedError("TODO: implement pivot calibration (see markdown above)")

p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list)



In [12]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(x=-0.42, y=-0.08, z=-49.83)
Internal Vector:
[[ -0.421]
 [ -0.084]
 [-49.828]]
estimated covariance (mm^2):
 [[ 0.001  0.     0.   ]
 [ 0.     0.001 -0.   ]
 [ 0.    -0.     0.001]]
estimated std-dev per axis (mm): [0.028 0.028 0.028]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [13]:
near_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points for this reference body
    vct3(-150, -150, 0),
    vct3(150, -150, 0),
    vct3(0, 150, 0),
    vct3(0, 0, 200)
]
F_near_approx = Frame(Rot(), vct3(300, 400, 525))   # TODO: Frame, placed close to (not inside) the test-pose cube

near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)


near_mb pass 1 done
near_mb pass 2 done
near_mb pass 3 done


Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [14]:
#thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.

test_poses = [
    # TODO: build a list of Frame(Rot.xyz(rx, ry, rz), vct3(300+tx, 300+ty, 300+tz))
    # from thetas/offsets above
    # Frame(Rot.xyz(0, 0, 0), vct3(300, 300, 300)),
    # Frame(Rot.xyz(0, , 0), vct3(300, 300, 300)),
]

rng = np.random.default_rng(0)

chosen = set()
while len(test_poses) < N_TEST_POSES:
    pick = (rng.choice(thetas), rng.choice(thetas), rng.choice(thetas),
            rng.choice(offsets), rng.choice(offsets), rng.choice(offsets))
    if pick in chosen:
        continue 
    chosen.add(pick)
    rx, ry, rz, tx, ty, tz = pick
    test_poses.append(Frame(Rot.xyz(rx, ry, rz), vct3(300 + tx, 300 + ty, 300 + tz)))

tip = ptr.get_actual_tip_position()
true_tip_local = vct3(tip.x, tip.y, tip.z)

test_results = []  # list of (F_t_nominal, error_mm)

predicted_positions = []
true_positions = []

for F_t_nominal in test_poses:
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")

    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])
    F_ptr = obs["ptr_mb"].frame
    F_near = obs["near_mb"].frame
    
    #   3. predict the tip position relative to near_placed using p_tip_est
    tip_predicted = F_near.inv() * (F_ptr * p_tip_est)

    #   4. compute the TRUE tip position relative to near_placed using
    F_real_true = near_placed.actual_placement.inv() * test_placed.actual_placement
    tip_true = F_real_true * true_tip_local
    
    predicted_positions.append(
    tip_predicted.vec.ravel()
    )

    true_positions.append(
        tip_true.vec.ravel()
    )
    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)
    #   5. record the error and append (F_t_nominal, error) to test_results
    error = np.linalg.norm(tip_predicted.vec.ravel() - tip_true.vec.ravel())
    test_results.append((F_t_nominal, error))

predicted_positions = np.array(predicted_positions)
true_positions = np.array(true_positions)

errors = []                                        
for i in range(len(test_results)):
    errors.append(test_results[i][1])
errors = np.array(errors)

print("test poses:  ", len(errors))
print("mean error (mm):  ", round(np.mean(errors)))
print("max error(mm):  ", round(np.max(errors)))

test poses:   100
mean error (mm):   1
max error(mm):   2


In [15]:
z_levels = [100, 200, 300, 400, 500]

print("z (mm) | poses | mean error (mm) | max error (mm)")
for z in z_levels:
    errors_at_z = []
    for i in range(len(test_results)):
        F_t_nominal = test_results[i][0]
        error = test_results[i][1]
        if F_t_nominal.p.vec.ravel()[2] == z:
            errors_at_z.append(error)

    if len(errors_at_z) == 0:
        print(z, "| 0 poses")
        continue
    errors_at_z = np.array(errors_at_z)
    print(z, "|", len(errors_at_z), "|", round(np.mean(errors_at_z), 3), "|", round(np.max(errors_at_z), 3))

z (mm) | poses | mean error (mm) | max error (mm)
100 | 23 | 0.88 | 1.986
200 | 12 | 0.922 | 1.544
300 | 27 | 0.954 | 1.809
400 | 14 | 0.75 | 1.285
500 | 24 | 1.007 | 1.687


## Achieved performance
Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [16]:
# TODO: summarize test_results -- e.g. mean/max error, and how many poses
# meet the assignment's ||delta p_t|| <= 2mm spec.

def summarize_tip_errors(error_norms, spec_mm=TIP_ERROR_SPEC_MM):
    error_norms = np.asarray(error_norms, dtype=float)
    return{
        'mean_mm': error_norms.mean(),
        'rms_mm': np.sqrt(np.mean(error_norms**2)),
        'max_mm': error_norms.max(),
        'std_mm': error_norms.std(),
        'worst_pose_index': int(error_norms.argmax()),
        'num_passing': int(np.sum(error_norms <= spec_mm)),
        'num_poses': len(error_norms),
    }


error_norms = [r[1] for r in test_results]
summary = summarize_tip_errors(error_norms)

print(f"Poses meeting {TIP_ERROR_SPEC_MM} mm spec: "
      f"{summary['num_passing']} /{summary['num_poses']}")
print(f"Mean error: {summary['mean_mm']: .3f} mm")
print(f"RMS error: {summary['rms_mm']: .3f} mm")
print(f"Max error: {summary['max_mm']: .3f} mm (pose {summary['worst_pose_index']})")
print(f"Std dev {summary['std_mm']: .3f} mm")

Poses meeting 2 mm spec: 100 /100
Mean error:  0.917 mm
RMS error:  0.998 mm
Max error:  1.986 mm (pose 68)
Std dev  0.392 mm


In [17]:
from scipy import stats
import numpy as np

# One-sided one-sample t-test against the 2 mm error specification
alpha = 0.05
spec_mm = TIP_ERROR_SPEC_MM

errors = np.asarray(errors, dtype=float)

n = len(errors)
mean_error = np.mean(errors)
std_error = np.std(errors, ddof=1)

# t statistic
t_stat = (
    mean_error - spec_mm
) / (
    std_error / np.sqrt(n)
)

# One-sided p-value for:
# H0: mean error >= 2 mm
# H1: mean error < 2 mm
p_value = stats.t.cdf(
    t_stat,
    df=n - 1
)

# 95% one-sided upper confidence bound for mean error
t_crit = stats.t.ppf(
    0.95,
    df=n - 1
)

upper_95 = (
    mean_error
    + t_crit * std_error / np.sqrt(n)
)

print(f"Number of poses: {n}")
print(f"Mean error: {mean_error:.3f} mm")
print(f"Standard deviation: {std_error:.3f} mm")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"95% upper confidence bound: {upper_95:.3f} mm")

if p_value < alpha:
    print(
        "Reject H0: the mean localization error is "
        "significantly below 2 mm at the 95% confidence level."
    )
else:
    print(
        "Fail to reject H0: there is not sufficient evidence "
        "that the mean localization error is below 2 mm."
    )

Number of poses: 100
Mean error: 0.917 mm
Standard deviation: 0.394 mm
t-statistic: -27.494
p-value: 0.000000
95% upper confidence bound: 0.983 mm
Reject H0: the mean localization error is significantly below 2 mm at the 95% confidence level.


In [18]:
from scipy.stats import linregress
import numpy as np

# Extract z-position from each test pose
z_values = np.array([
    F.p.vec.ravel()[2]
    for F, error in test_results
])

# Extract corresponding localization errors
errors = np.array([
    error
    for F, error in test_results
])

slope, intercept, r_value, p_value, std_err = linregress(
    z_values,
    errors
)

print(f"Slope:   {slope:.5f} mm error / mm z")
print(f"R^2:     {r_value**2:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("There is a statistically significant relationship between z-position and localization error.")
else:
    print("There is not a statistically significant relationship between z-position and localization error.")
    

from scipy.stats import linregress

slope, intercept, r_value, p_value, std_err = linregress(
    z_values,
    errors
)

print("Slope:", slope)
print("R^2:", r_value**2)
print("p-value:", p_value)

Slope:   0.00017 mm error / mm z
R^2:     0.0039
p-value: 0.535147
There is not a statistically significant relationship between z-position and localization error.
Slope: 0.00016810344322586957
R^2: 0.003936882327479642
p-value: 0.5351466820234836


### Unit test for refine_marker_body

In [19]:
test_body_positions = [vct3(0, 0, 0), vct3(0, 50, 0), vct3(0, 50, 0), vct3(0, 0, 50)]
test_mb = simulation.define_marker_body("test_mb", test_body_positions)
test_placed = simulation.place_marker_body(test_mb, Frame.eye())

# marker positions before refinement
before = []
for a in test_mb.nominal_marker_positions:
    before.append(np.array(a.vec).reshape(3))

#the refinement function being called
refine_marker_body(trk, test_placed, n_readings=30, passes=3)

#marker positions after refinement
after = []
for a in test_mb.nominal_marker_positions:
    after.append(np.array(a.vec).reshape(3))

#distance between positions before and after 
shifts = []
for i in range(len(before)):
    distance_moved = float(np.linalg.norm(after[i] - before[i]))
    shifts.append(distance_moved)

print("How for each marker moved (mm):", shifts)
print("Largest shift (mm):", max(shifts))

if max(shifts) < 5.0:
    print("PASSED: marker stayed closed to their original positions")
else:
    print("FAILED: a marker moved further than expected")

test_mb pass 1 done
test_mb pass 2 done
test_mb pass 3 done
How for each marker moved (mm): [0.8771850016165704, 0.3918059965921604, 1.0753332288907853, 0.38343089075621156]
Largest shift (mm): 1.0753332288907853
PASSED: marker stayed closed to their original positions


### Unit test for compute_pivot_calibration

In [20]:
from scipy import stats

n_trials = 30
trial_errors = []

for i in range(n_trials):
    
    true_p_tip = vct3(*np.random.uniform(-50, 50, size=3))

    F_ptr_test = []
    F_cal_test = [] 
    sensed_test = []

    #10 fake observations per trial
    for j in range(10):
        R = Rot.rand_xyz(np.pi/4, np.pi/4, np.pi)
        p_ptr = np.random.uniform(-500, 500, 3)
        F_ptr_test.append(Frame(R, vct3(*p_ptr)))
        F_cal_test.append(Frame.eye())

        #tip position in the cal object frame
        s_j = R.matrix @ true_p_tip.vec.ravel() + p_ptr
        sensed_test.append(vct3(*s_j))

    #compute the estimated tip position using the pivot calibration function
    p_tip_est_test, _ = compute_pivot_calibration(F_ptr_test, F_cal_test, sensed_test)
    #difference between the estimated tip position and the true tip position
    err = np.linalg.norm(p_tip_est_test.vec.ravel() - true_p_tip.vec.ravel())
    trial_errors.append(err)

trial_errors = np.array(trial_errors)
mean_err = np.mean(trial_errors)
std_err = np.std(trial_errors, ddof=1)

print(f"Mean error: {mean_err:.3e} mm, Std dev: {std_err:.3e} mm")

tolerance = 1e-9
if mean_err < tolerance:
    print(f"PASSED: mean error is below {tolerance} mm")
else:
    print(f"FAILED: mean error is above {tolerance} mm")

Mean error: 1.303e-14 mm, Std dev: 6.225e-15 mm
PASSED: mean error is below 1e-09 mm


In [21]:
### Diagnostic sweep: run Steps 4-8 multiple times, log everything
N_RUNS = 15
sweep_results = []

for run in range(N_RUNS):
    # --- Steps 4-6: collect calibration observations ---
    F_ptr_list_s = []
    F_cal_list_s = []
    sensed_tip_list_s = []
    for j in range(N_OBSERVATIONS):
        R_desired = Rot.rand_xyz(np.pi/4, np.pi/4, np.pi)
        hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
        obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
        sensed = simulation.sense_tip(ptr, hg)
        F_ptr_list_s.append(obs["ptr_mb"].frame)
        F_cal_list_s.append(obs["cal_mb"].frame)
        sensed_tip_list_s.append(sensed)

    M_valid = sum(1 for s in sensed_tip_list_s if s is not None)

    # --- Step 7: calibrate, and capture condition number ---
    p_tip_run, cov_run = compute_pivot_calibration(F_ptr_list_s, F_cal_list_s, sensed_tip_list_s)

    # recompute A just to get its condition number (mirrors what's inside the function)
    A_blocks = []
    for i in range(len(F_ptr_list_s)):
        if sensed_tip_list_s[i] is None:
            continue
        F_j = F_cal_list_s[i].inv() * F_ptr_list_s[i]
        A_blocks.append(F_j.R.matrix)
    A_cond = np.linalg.cond(np.vstack(A_blocks))

    # --- Step 8: run full test-pose validation ---
    run_errors = []
    run_z = []
    for F_t_nominal in test_poses:
        test_placed_s = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
        obs = simulation.sample_marker_bodies(trk, [test_placed_s, near_placed])
        F_ptr_s = obs["ptr_mb"].frame
        F_near_s = obs["near_mb"].frame
        tip_predicted = F_near_s.inv() * (F_ptr_s * p_tip_run)
        F_real_true = near_placed.actual_placement.inv() * test_placed_s.actual_placement
        tip_true = F_real_true * true_tip_local
        err = np.linalg.norm(tip_predicted.vec.ravel() - tip_true.vec.ravel())
        run_errors.append(err)
        run_z.append(F_t_nominal.p.vec.ravel()[2])

    run_errors = np.array(run_errors)
    run_z = np.array(run_z)
    num_passing = int(np.sum(run_errors <= TIP_ERROR_SPEC_MM))

    sweep_results.append({
        "run": run,
        "M_valid": M_valid,
        "A_cond": A_cond,
        "num_passing": num_passing,
        "mean_error": run_errors.mean(),
        "max_error": run_errors.max(),
        "errors": run_errors,
        "z": run_z,
    })

    print(f"Run {run:2d}: M={M_valid}/{N_OBSERVATIONS}  cond(A)={A_cond:7.2f}  "
          f"pass={num_passing:3d}/100  mean_err={run_errors.mean():.3f}mm  max_err={run_errors.max():.3f}mm")

Run  0: M=48/50  cond(A)=   1.00  pass= 99/100  mean_err=0.932mm  max_err=2.010mm
Run  1: M=48/50  cond(A)=   1.00  pass=100/100  mean_err=0.916mm  max_err=1.911mm
Run  2: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.921mm  max_err=1.783mm
Run  3: M=49/50  cond(A)=   1.00  pass= 99/100  mean_err=0.939mm  max_err=2.036mm
Run  4: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.922mm  max_err=1.896mm
Run  5: M=50/50  cond(A)=   1.00  pass=100/100  mean_err=0.922mm  max_err=1.936mm
Run  6: M=46/50  cond(A)=   1.00  pass=100/100  mean_err=0.911mm  max_err=1.822mm
Run  7: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.931mm  max_err=1.972mm
Run  8: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.915mm  max_err=1.910mm
Run  9: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.921mm  max_err=1.948mm
Run 10: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.929mm  max_err=1.887mm
Run 11: M=46/50  cond(A)=   1.00  pass=100/100  mean_err=0.919mm  max_err=1.952mm
Run 12: M=50/50 

In [22]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    body = placed.body
    for i in range(passes):
        readings = []
        for j in range(n_readings):
            obs = simulation.sample_marker_bodies(trk, [placed])[body.name]
            F_inv = obs.frame.inv()
            this_reading = []
            for m in obs.markers:
                b = m.p
                a = F_inv * b
                this_reading.append(np.array(a.vec).reshape(3))
            readings.append(this_reading)
        averages = np.mean(readings, axis=0)
        body.set_marker_positions([vct3(*row) for row in averages])


### Full diagnostic sweep: re-design geometry + refine + Steps 4-8, every run independent
N_RUNS = 10
sweep_results = []

for run in range(N_RUNS):
    ptr_mb_s = simulation.define_marker_body(f"ptr_mb_{run}", ptr_local_positions)
    cal_mb_s = simulation.define_marker_body(f"cal_mb_{run}", cal_local_positions)
    near_mb_s = simulation.define_marker_body(f"near_mb_{run}", near_local_positions)

    cal_placed_s = simulation.place_marker_body(cal_mb_s, F_cal_approx)
    ptr_placed_s = simulation.place_marker_body(ptr_mb_s, Frame.eye(), precision="precise")
    near_placed_s = simulation.place_marker_body(near_mb_s, F_near_approx)

    ptr_s = simulation.define_pointer(f"ptr_{run}", ptr_tip_approx, ptr_mb_s)
    trk_s = simulation.create_tracker(f"trk_{run}", F_tracker_approx)

    refine_marker_body(trk_s, ptr_placed_s, n_readings=200, passes=10)
    refine_marker_body(trk_s, cal_placed_s, n_readings=200, passes=10)
    refine_marker_body(trk_s, near_placed_s, n_readings=200, passes=10)

    F_ptr_list_s, F_cal_list_s, sensed_tip_list_s = [], [], []
    for j in range(N_OBSERVATIONS):
        R_desired = Rot.rand_xyz(np.pi/2, np.pi/2, np.pi)
        hg = simulation.hand_guide_pointer(ptr_s, ptr_placed_s, cal_placed_s, R_desired)
        obs = simulation.sample_marker_bodies(trk_s, [ptr_placed_s, cal_placed_s])
        sensed = simulation.sense_tip(ptr_s, hg)
        F_ptr_list_s.append(obs[f"ptr_mb_{run}"].frame)
        F_cal_list_s.append(obs[f"cal_mb_{run}"].frame)
        sensed_tip_list_s.append(sensed)

    M_valid = sum(1 for s in sensed_tip_list_s if s is not None)
    p_tip_run, cov_run = compute_pivot_calibration(F_ptr_list_s, F_cal_list_s, sensed_tip_list_s)

    A_blocks = []
    for i in range(len(F_ptr_list_s)):
        if sensed_tip_list_s[i] is None:
            continue
        F_j = F_cal_list_s[i].inv() * F_ptr_list_s[i]
        A_blocks.append(F_j.R.matrix)
    A_cond = np.linalg.cond(np.vstack(A_blocks))

    tip_s = ptr_s.get_actual_tip_position()
    true_tip_local_s = vct3(tip_s.x, tip_s.y, tip_s.z)

    run_errors = []
    for F_t_nominal in test_poses:
        test_placed_s = simulation.place_marker_body(ptr_mb_s, F_t_nominal, precision="precise")
        obs = simulation.sample_marker_bodies(trk_s, [test_placed_s, near_placed_s])
        F_ptr_s2 = obs[f"ptr_mb_{run}"].frame
        F_near_s = obs[f"near_mb_{run}"].frame
        tip_predicted = F_near_s.inv() * (F_ptr_s2 * p_tip_run)
        F_real_true = near_placed_s.actual_placement.inv() * test_placed_s.actual_placement
        tip_true = F_real_true * true_tip_local_s
        run_errors.append(np.linalg.norm(tip_predicted.vec.ravel() - tip_true.vec.ravel()))

    run_errors = np.array(run_errors)
    num_passing = int(np.sum(run_errors <= TIP_ERROR_SPEC_MM))

    sweep_results.append({
        "run": run, "M_valid": M_valid, "A_cond": A_cond,
        "num_passing": num_passing, "mean_error": run_errors.mean(),
        "max_error": run_errors.max(),
    })

    print(f"Run {run:2d}: M={M_valid}/{N_OBSERVATIONS}  cond(A)={A_cond:7.2f}  "
          f"pass={num_passing:3d}/100  mean_err={run_errors.mean():.3f}mm  max_err={run_errors.max():.3f}mm")

all_means = [r["mean_error"] for r in sweep_results]
all_passes = [r["num_passing"] for r in sweep_results]
print(f"\nAcross {N_RUNS} fully independent trials:")
print(f"  Mean error: {np.mean(all_means):.3f}mm (range {min(all_means):.3f}-{max(all_means):.3f}mm)")
print(f"  Pass rate: {np.mean(all_passes):.1f}/100 (range {min(all_passes)}-{max(all_passes)})")

Run  0: M=48/50  cond(A)=   1.00  pass=100/100  mean_err=0.859mm  max_err=1.410mm
Run  1: M=48/50  cond(A)=   1.00  pass=100/100  mean_err=0.791mm  max_err=1.623mm
Run  2: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.671mm  max_err=1.279mm
Run  3: M=47/50  cond(A)=   1.00  pass= 77/100  mean_err=1.562mm  max_err=2.948mm
Run  4: M=49/50  cond(A)=   1.00  pass=100/100  mean_err=0.792mm  max_err=1.432mm
Run  5: M=46/50  cond(A)=   1.00  pass= 98/100  mean_err=1.233mm  max_err=2.292mm
Run  6: M=45/50  cond(A)=   1.00  pass= 58/100  mean_err=1.906mm  max_err=3.620mm
Run  7: M=49/50  cond(A)=   1.00  pass= 92/100  mean_err=1.166mm  max_err=2.292mm
Run  8: M=48/50  cond(A)=   1.00  pass= 94/100  mean_err=1.329mm  max_err=2.237mm
Run  9: M=46/50  cond(A)=   1.00  pass=100/100  mean_err=0.659mm  max_err=1.295mm

Across 10 fully independent trials:
  Mean error: 1.097mm (range 0.659-1.906mm)
  Pass rate: 91.9/100 (range 58-100)
